# MZM 소자 측정 데이터 분석 및 시각화 파이프라인

본 노트북은 **MZM(Mach-Zehnder Modulator)** 소자로부터 측정된 raw 데이터를 자동으로 파싱하고, 핵심 특성 파라미터(IL, ER, Vpi 등)를 분석 및 추출하여 **Wafer Map** 및 **Box Plot** 형태로 시각화하는 전체 공정을 포함하고 있습니다.
# 이 노트북을 실행하기 전 필요한 라이브러리 설치
# !pip install numpy pandas matplotlib scipy

## 📌 데이터 분석 파이프라인 개요
1. **데이터 전처리 & 파싱**: Raw 데이터 로드 및 정제 (`data_parser`,`plot`)
2. **소자 특성 피팅 & 분석**: 물리적 모델 기반 피팅 및 파라미터 추출 (`zoom`, `flatting`,`Fitting`, `Phase shift - V`, `VpiL`)
3. **핵심 지표 산출**: 삽입 손실(IL), 소멸비(ER), 구동 전압(Vpi) 분석 (`IL_Analysis`, `ER_Analysis`, `VpiL_Analysis`)
4. **시각화 및 요약**: 데이터 시각화 및 최종 보고서 내보내기 ( `combine_plot`,  `export_summary`)

In [17]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from datetime import datetime


def parse_wafer_data(data_path, target_wafers):
    """지정된 폴더(data_path)를 탐색하여 XML 데이터를 파싱하고 다이(Die)별 데이터와 측정 날짜를 반환합니다."""

    # os.walk를 사용하여 폴더 내의 모든 파일을 재귀적으로(하위 폴더까지) 탐색합니다.
    for root_dir, dirs, files in os.walk(data_path):
        for file_name in files:
            if not file_name.lower().endswith('.xml'):
                continue

            # 파일 경로 전체를 문자열로 만듭니다.
            file_path = os.path.join(root_dir, file_name)

            # 파일 경로나 이름에 타겟 웨이퍼(D07, D08 등)가 포함되어 있는지 확인
            if not any(w in file_path for w in target_wafers):
                continue

            try:
                # 일반 폴더에 있는 파일이므로 open() 함수로 바로 엽니다.
                with open(file_path, 'r', encoding='utf-8') as f:
                    tree = ET.parse(f)
                    root = tree.getroot()

                    info = root.find('.//TestSiteInfo')
                    if info is None: continue

                    test_site = info.get('TestSite', '')
                    if 'LMZO' in test_site.upper():
                        band, wl_min, wl_max, tgt_wl = 'LMZO', 1260.0, 1360.0, 1310.0
                    elif 'LMZC' in test_site.upper():
                        band, wl_min, wl_max, tgt_wl = 'LMZC', 1520.0, 1580.0, 1550.0
                    else:
                        continue

                    die_c = int(info.get('DieRow', 0)) if info.get('DieRow') else 0
                    die_r = int(info.get('DieColumn', 0)) if info.get('DieColumn') else 0

                    # 웨이퍼 아이디는 파일명이나 폴더명(경로)에 포함되어 있는지 확인하여 추출
                    wafer_id = next((w for w in target_wafers if w in file_path), "Unknown")

                    # =========================================================
                    # 날짜 정보 추출 및 'YYYYMMDD' 변환 로직 (기존 로직 유지)
                    date_raw = info.get('Date') or root.get('Date') or root.get('CreationDate')
                    if not date_raw:
                        date_elem = root.find('.//Date') or root.find('.//CreationDate')
                        if date_elem is not None:
                            date_raw = date_elem.text

                    date_str = "Unknown_Date"
                    if date_raw:
                        date_clean = date_raw.split('.')[0].strip()

                        date_formats = [
                            "%a %b %d %H:%M:%S %Y",
                            "%Y-%m-%d %H:%M:%S",
                            "%Y-%m-%d",
                            "%Y-%m-%dT%H:%M:%S",
                            "%Y/%m/%d %H:%M:%S"
                        ]

                        for fmt in date_formats:
                            try:
                                dt = datetime.strptime(date_clean, fmt)
                                date_str = dt.strftime("%Y%m%d")
                                break
                            except ValueError:
                                continue

                        if date_str == "Unknown_Date":
                            date_str = "".join(filter(str.isalnum, date_clean))
                    # =========================================================

                    sweeps = root.findall('.//WavelengthSweep')
                    if not sweeps or len(sweeps) < 2: continue

                    ref_data = None
                    bias_list = []

                    for i, sweep in enumerate(sweeps):
                        l_elem, il_elem = sweep.find('L'), sweep.find('IL')
                        if l_elem is None or il_elem is None: continue

                        l_data = np.array(list(map(float, l_elem.text.split(','))))
                        il_data = np.array(list(map(float, il_elem.text.split(','))))
                        bias_str = sweep.get('DCBias', '')

                        if i == len(sweeps) - 1:
                            ref_data = {'wl': l_data, 'il': il_data, 'label': 'REF'}
                        else:
                            try:
                                bias_val = float(bias_str)
                                label = f'{bias_val}V'
                            except:
                                bias_val = None
                                label = f'Bias: {bias_str}'
                            bias_list.append({'bias': bias_val, 'wl': l_data, 'il': il_data, 'label': label})

                    if ref_data is None: continue

                    yield {
                        'wafer_id': wafer_id, 'band': band, 'die_c': die_c, 'die_r': die_r,
                        'wl_min': wl_min, 'wl_max': wl_max, 'target_wl': tgt_wl,
                        'date': date_str,
                        'ref_data': ref_data, 'bias_data_list': bias_list
                    }
            except Exception as e:
                print(f"[{file_path}] 파싱 에러: {e}")

### 🖼️ [2] 단일 데이터 시각화 (Plot)
* **파일명**: `plot.py`
* **설명**: 개별 소자 혹은 특정 다이(Die)의 파장별 투과 특성 곡선 및 전압 특성 곡선을 시각화합니다.

In [18]:
import matplotlib
import matplotlib.pyplot as plt

# Jupyter에서도 GUI 충돌 방지
matplotlib.use("Agg")

def _draw_and_save_plot(d, base_save_dir):
    fig, ax = plt.subplots(figsize=(10, 6))

    try:
        # Bias 데이터 플롯
        for b in d['bias_data_list']:
            wl = np.asarray(b['wl'])
            il = np.asarray(b['il'])

            if len(wl) == 0 or len(il) == 0:
                continue

            if np.isnan(il).all():
                print(f"NaN only: {d['wafer_id']} C{d['die_c']} R{d['die_r']} {d['band']}")
                continue

            ax.plot(wl, il, label=b['label'], linewidth=2.5)

        # Reference 데이터 플롯
        ref_wl = np.asarray(d['ref_data']['wl'])
        ref_il = np.asarray(d['ref_data']['il'])

        if len(ref_wl) > 0 and not np.isnan(ref_il).all():
            ax.plot(
                ref_wl, ref_il,
                label=d['ref_data']['label'],
                linewidth=2.5, color='black',
                alpha=0.8, linestyle='--'
            )

        # 그래프 서식 설정
        ax.set_title(
            f"Wafer: {d['wafer_id']} / Coord: ({d['die_c']}, {d['die_r']}) / Band: {d['band']}",
            fontsize=18, fontweight='bold', pad=15
        )
        ax.set_xlabel("Wavelength (nm)", fontsize=16, fontweight='bold')
        ax.set_ylabel("Transmission (dB)", fontsize=16, fontweight='bold')
        ax.set_ylim(-65, 5)
        ax.tick_params(axis='both', labelsize=13, width=2)
        ax.legend(loc='best', prop={'size': 12, 'weight': 'bold'})
        ax.grid(True, linestyle='--', alpha=0.6, linewidth=1)

        # 축 두께 설정
        for spine in ax.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        # 저장 경로 및 파일명 설정
        date_str = d.get('date', 'Unknown_Date')
        w_dir = os.path.join(base_save_dir, d['wafer_id'], date_str)
        os.makedirs(w_dir, exist_ok=True)

        save_filename = f"{d['wafer_id']}_C{d['die_c']}_R{d['die_r']}_{d['band']}_Raw.png"

        # 이미지 저장
        fig.savefig(os.path.join(w_dir, save_filename), bbox_inches='tight', dpi=150)

        return save_filename

    finally:
        # 메모리 누수 방지를 위해 figure 닫기
        plt.close(fig)

# 실행 영역
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱 시작")

# 주의: parse_wafer_data 함수는 다른 셀이나 파일에서 미리 import 되어 있어야 합니다.
parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))

total_tasks = len(parsed_data_list)
print(f"▶ 총 {total_tasks}개 그래프")

for i, d in enumerate(parsed_data_list, 1):
    _draw_and_save_plot(d, base_save_dir)

    # 진행 상황 출력
    if i % 50 == 0:
        print(f"진행: {i}/{total_tasks}")

print(f"✅ 총 {total_tasks}개 플롯 저장 완료")

▶ 데이터 파싱 시작
▶ 총 98개 그래프
진행: 50/98
✅ 총 98개 플롯 저장 완료


### 📐 [3] 데이터 평탄화 (Flatting)
* **파일명**: `flatting.py`
* **설명**: 측정 장비의 광학적 baseline 감쇄나 노이즈를 제거하기 위해 전력(Power) 또는 투과율 데이터를 평탄화(Flattening)하는 전처리 작업을 수행합니다.

In [19]:
from scipy.signal import find_peaks

def _draw_and_save_flat(d, base_save_dir):

    try:

        m = (
            (d['ref_data']['wl'] >= d['wl_min']) &
            (d['ref_data']['wl'] <= d['wl_max'])
        )

        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        if len(v_ref_wl) < 4:
            return None

        if np.isnan(v_ref_il).all():
            return None

        poly = np.poly1d(np.polyfit( v_ref_wl, v_ref_il, 3))

        fig, ax = plt.subplots(figsize=(10, 6))

        curve_count = 0

        for b in d['bias_data_list']:

            m_b = (
                (b['wl'] >= d['wl_min']) &
                (b['wl'] <= d['wl_max'])
            )

            v_wl = b['wl'][m_b]
            v_il = b['il'][m_b]

            if len(v_wl) == 0:
                continue

            if len(v_il) == 0:
                continue

            if np.isnan(v_il).all():
                continue

            flat_il = v_il - poly(v_wl)

            peaks, _ = find_peaks(flat_il,
                prominence=3.0,distance=30)

            if len(peaks) >= 2:

                linear_fit = np.poly1d(
                    np.polyfit( v_wl[peaks],flat_il[peaks],1)
                )

                flat_il = flat_il - linear_fit(v_wl)

            ax.plot(v_wl,flat_il,label=b['label'],
                alpha=0.8,linewidth=2.5
            )

            curve_count += 1

        ref_flat = v_ref_il - poly(v_ref_wl)

        ax.plot(v_ref_wl,ref_flat,label='REF',
            linewidth=2.5,color='black',alpha=0.8,
            linestyle='--')

        curve_count += 1

        if curve_count == 0:
            plt.close(fig)
            return None

        ax.set_title(
            f"Wafer: {d['wafer_id']} / "
            f"Coord: ({d['die_c']}, {d['die_r']}) / "
            f"Band: {d['band']} Flattened",
            fontsize=18,fontweight='bold',pad=15
        )

        ax.set_xlabel('Wavelength (nm)',fontsize=16,
            fontweight='bold'
        )

        ax.set_ylabel('Transmission (dB)',
            fontsize=16,fontweight='bold'
        )

        ax.axhline(0,color='gray',linestyle='--',
            linewidth=2
        )

        ax.set_xlim(d['wl_min'],d['wl_max']
        )
        ax.set_ylim(-65, 5)
        ax.grid(True, linestyle='--',alpha=0.6,
            linewidth=1
        )

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        handles, labels = ax.get_legend_handles_labels()

        if len(handles) > 0:
            ax.legend(loc='best',prop={'size': 12,
                    'weight': 'bold'
                }
            )

        fig.tight_layout()

        date_str = d.get('date','Unknown_Date'
        )

        save_dir = os.path.join(
            base_save_dir,d['wafer_id'],date_str
        )

        os.makedirs(save_dir,exist_ok=True
        )

        filename = (
            f"{d['wafer_id']}"f"_C{d['die_c']}"
            f"_R{d['die_r']}"f"_{d['band']}"
            f"_Flat.png"
        )

        fig.savefig(
            os.path.join(save_dir, filename),
            bbox_inches='tight',dpi=150
        )

        plt.close(fig)

        return filename

    except Exception as e:

        print(f"오류 발생: "
            f"{d['wafer_id']} "f"C{d['die_c']} "
            f"R{d['die_r']} "f"{d['band']} "f"-> {e}"
        )

        return None
# 실행부

zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("▶ 데이터 파싱을 시작합니다...")

parsed_data_list = list(
    parse_wafer_data(zip_path,target_wafers
    )
)

total_items = len(parsed_data_list)

print(f"▶ 기본 평탄화 플롯 생성 "f"({total_items}개)"
)

success_count = 0

for i, d in enumerate(parsed_data_list, 1):

    result = _draw_and_save_flat(d,base_save_dir
    )

    if result is not None:
        success_count += 1

    if i % 50 == 0:
        print(f"진행: {i}/{total_items}")

print(f"✅ 기본 평탄화 저장 완료 "f"(총 {success_count}개)"
)

▶ 데이터 파싱을 시작합니다...
▶ 기본 평탄화 플롯 생성 (98개)
진행: 50/98
✅ 기본 평탄화 저장 완료 (총 98개)


### 🔍 [4] 특정 영역 확대 분석 (Zoom)
* **파일명**: `zoom.py`
* **설명**: 주요 딥(Dip) 구간이나 피크(Peak) 구간 등 정밀 분석이 필요한 특정 파장/전압 영역을 확대하여 시각화합니다.

In [20]:
from scipy.signal import find_peaks

def _draw_and_save_zoom_only(d, base_save_dir):

    try:

        m = (
            (d['ref_data']['wl'] >= d['wl_min']) &
            (d['ref_data']['wl'] <= d['wl_max'])
        )

        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        if len(v_ref_wl) < 4:
            return None

        if np.isnan(v_ref_il).all():
            return None

        poly = np.poly1d(
            np.polyfit(v_ref_wl,v_ref_il,3)
        )

        z_min = d['wl_min']
        z_max = d['wl_max']

        t_bias = next(
            (
                b
                for b in d['bias_data_list']
                if b['bias'] == -2.0),
            d['bias_data_list'][0]
            if d['bias_data_list']
            else None
        )

        if t_bias is not None:

            mt = (
                (t_bias['wl'] >= d['wl_min']) &
                (t_bias['wl'] <= d['wl_max'])
            )

            wt = t_bias['wl'][mt]
            it = t_bias['il'][mt]

            if len(wt) > 0:

                peaks, _ = find_peaks(
                    it - poly(wt),
                    prominence=3.0,distance=30
                )

                if len(peaks) >= 2:

                    cent = (wt[peaks[:-1]]
                        + wt[peaks[1:]]) / 2.0

                    band_str = str(
                        d.get('band', '')).upper()

                    if 'O' in band_str:
                        target_wl = d.get(
                            'target_wl',1310.0
                        )
                    else:
                        target_wl = d.get(
                            'target_wl',1550.0
                        )

                    idx = np.argmin(
                        np.abs(cent - target_wl
                        )
                    )

                    if idx + 1 < len(peaks):

                        z_min = (wt[peaks[idx]]- 0.5
                        )

                        z_max = (wt[peaks[idx + 1]]
                            + 0.5
                        )

        fig, ax = plt.subplots(
            figsize=(10, 6)
        )

        curve_count = 0

        for b in d['bias_data_list']:

            mb = (
                (b['wl'] >= d['wl_min']) &
                (b['wl'] <= d['wl_max'])
            )

            wb = b['wl'][mb]
            ib = b['il'][mb]

            if len(wb) == 0:
                continue

            if np.isnan(ib).all():
                continue

            flat = ib - poly(wb)

            peaks, _ = find_peaks(
                flat,prominence=3.0,distance=30
            )

            if len(peaks) >= 2:

                linear_fit = np.poly1d(
                    np.polyfit(wb[peaks],flat[peaks],1
                    )
                )

                flat -= linear_fit(wb)

            ax.plot(wb,flat,label=b['label'],
                alpha=0.8,linewidth=2.5
            )

            curve_count += 1

        if curve_count == 0:
            plt.close(fig)
            return None

        ax.axhline(
            0,color='gray',linestyle='--',linewidth=2
        )

        ax.set_xlim(z_min,z_max)

        ax.set_title(
            f"Wafer: {d['wafer_id']} / "
            f"Coord: ({d['die_c']}, {d['die_r']}) / "
            f"Band: {d['band']} Zoom Only",
            fontsize=18,fontweight='bold',pad=15
        )

        ax.set_xlabel('Wavelength (nm)',
            fontsize=16,fontweight='bold'
        )

        ax.set_ylabel('Transmission (dB)',
            fontsize=16,fontweight='bold'
        )

        ax.grid(True,
            linestyle='--',alpha=0.6,linewidth=1
        )

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        handles, labels = (
            ax.get_legend_handles_labels()
        )

        if len(handles) > 0:

            ax.legend(
                loc='best',
                prop={'size': 12,'weight': 'bold'
                }
            )

        fig.tight_layout()

        date_str = d.get(
            'date','Unknown_Date'
        )

        save_dir = os.path.join(
            base_save_dir,d['wafer_id'],date_str
        )

        os.makedirs(
            save_dir,exist_ok=True
        )

        filename = (
            f"{d['wafer_id']}"f"_C{d['die_c']}"
            f"_R{d['die_r']}"f"_{d['band']}"f"_Zoom.png"
        )

        fig.savefig(
            os.path.join(save_dir,filename
            ),bbox_inches='tight',dpi=150
        )

        plt.close(fig)

        return filename

    except Exception as e:

        print(
            f"오류 발생: "
            f"{d['wafer_id']} "
            f"C{d['die_c']} "
            f"R{d['die_r']} "
            f"{d['band']} "
            f"-> {e}"
        )

        return None

# 실행부

zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = ['D07','D08','D23','D24'
]

print("▶ 데이터 파싱을 시작합니다...")

parsed_data_list = list(
    parse_wafer_data(zip_path,target_wafers
    )
)

total_items = len(parsed_data_list)

print(
    f"▶ 줌 전용 플롯 생성 "f"({total_items}개)"
)

success_count = 0

for i, d in enumerate(parsed_data_list,1):

    result = _draw_and_save_zoom_only(
        d,base_save_dir
    )

    if result is not None:
        success_count += 1

    if i % 50 == 0:

        print(f"진행: "f"{i}/{total_items}"
        )

print(
    f"✅ 줌 전용 저장 완료 "
    f"(총 {success_count}개)"
)

▶ 데이터 파싱을 시작합니다...
▶ 줌 전용 플롯 생성 (98개)
진행: 50/98
✅ 줌 전용 저장 완료 (총 98개)


### 🔮 [5] 데이터 피팅 (Curve Fitting)
* **파일명**: `Fitting.py`
* **설명**: MZM의 인텐시티 변조 특성 곡선(주로 정현파 형태)을 이론적 수식 모델에 맞춰 커브 피팅(Curve Fitting)을 수행하고 주요 상수를 도출합니다.

In [21]:
from scipy.signal import find_peaks, savgol_filter

def _draw_and_save_zoomed_plot(d, base_save_dir):

    try:

        m = (
            (d['ref_data']['wl'] >= d['wl_min']) &
            (d['ref_data']['wl'] <= d['wl_max'])
        )

        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        if len(v_ref_wl) < 31:
            return None

        if np.isnan(v_ref_il).all():
            return None

        sm_ref = savgol_filter(v_ref_il,31,3
        )

        poly = np.poly1d(np.polyfit(v_ref_wl,sm_ref,3
            )
        )

        y_fitted = poly(v_ref_wl)

        ss_res = np.sum((sm_ref - y_fitted) ** 2)

        ss_tot = np.sum((sm_ref - np.mean(sm_ref)) ** 2
        )

        if ss_tot == 0:
            r_squared = 1.0
        else:
            r_squared = 1 - (ss_res / ss_tot)

        fig, ax = plt.subplots(figsize=(10, 6))

        z_min = d['wl_min']
        z_max = d['wl_max']

        tgt_bias = next(
            (
                b
                for b in d['bias_data_list']
                if b['bias'] == -2.0
            ),
            d['bias_data_list'][0]
            if d['bias_data_list']
            else None
        )

        if tgt_bias is not None:

            m_t = (
                (tgt_bias['wl'] >= d['wl_min']) &
                (tgt_bias['wl'] <= d['wl_max'])
            )

            w_t = tgt_bias['wl'][m_t]
            i_t = tgt_bias['il'][m_t]

            if len(w_t) >= 31:

                flat_t = (
                    savgol_filter(i_t, 31, 3)
                    - poly(w_t)
                )

                peaks, _ = find_peaks(
                    flat_t,prominence=3.0,distance=30
                )

                if len(peaks) >= 2:

                    centers = (w_t[peaks[:-1]]
                        + w_t[peaks[1:]]) / 2.0

                    band_str = str(
                        d.get('band', '')).upper()

                    if 'O' in band_str:
                        target_wl = d.get(
                            'target_wl',1310.0
                        )
                    else:
                        target_wl = d.get(
                            'target_wl',1550.0
                        )

                    idx = np.argmin(np.abs(
                            centers - target_wl)
                    )

                    if idx + 1 < len(peaks):

                        z_min = (
                            w_t[peaks[idx]]- 0.5
                        )

                        z_max = (
                            w_t[peaks[idx + 1]]
                            + 0.5
                        )

        curve_count = 0

        for b in d['bias_data_list']:

            m_b = (
                (b['wl'] >= d['wl_min']) &
                (b['wl'] <= d['wl_max'])
            )

            v_wl = b['wl'][m_b]
            v_il = b['il'][m_b]

            if len(v_wl) < 31:
                continue

            if np.isnan(v_il).all():
                continue

            flat_il = (
                savgol_filter(
                    v_il,31,3
                )- poly(v_wl)
            )

            peaks, _ = find_peaks(
                flat_il,prominence=3.0,distance=30
            )

            if len(peaks) >= 2:

                linear_fit = np.poly1d(
                    np.polyfit(
                        v_wl[peaks],flat_il[peaks],1
                    )
                )

                flat_il -= linear_fit(v_wl)

            ax.plot(
                v_wl,flat_il,label=b['label'],
                linewidth=2.5,alpha=0.8
            )

            curve_count += 1

        ax.plot(
            v_ref_wl,sm_ref - y_fitted,
            label=f'REF (R²={r_squared:.4f})',
            color='black',
            linewidth=2.5,linestyle='--',zorder=10
        )

        curve_count += 1

        if curve_count == 0:
            plt.close(fig)
            return None

        ax.set_title(
            f"Wafer: {d['wafer_id']} / "
            f"Coord: ({d['die_c']}, {d['die_r']}) / "
            f"Band: {d['band']}\n"
            f"Smoothed, Flattened & Zoomed",
            fontsize=18,fontweight='bold',pad=15
        )

        ax.set_xlabel(
            'Wavelength (nm)',
            fontsize=16,fontweight='bold'
        )

        ax.set_ylabel(
            'Transmission (dB)',
            fontsize=16,fontweight='bold'
        )

        ax.axhline(
            0,color='gray',
            linestyle='--',alpha=0.6,linewidth=2
        )

        ax.set_xlim(
            z_min,z_max
        )

        ax.grid(
            True,linestyle='--',alpha=0.6,linewidth=1
        )

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        handles, labels = (
            ax.get_legend_handles_labels()
        )

        if len(handles) > 0:

            ax.legend(
                loc='best',
                prop={
                    'size': 12,'weight': 'bold'
                }
            )

        fig.tight_layout()

        date_str = d.get(
            'date','Unknown_Date'
        )

        save_dir = os.path.join(
            base_save_dir,d['wafer_id'],date_str
        )

        os.makedirs(
            save_dir,exist_ok=True
        )

        filename = (
            f"{d['wafer_id']}"
            f"_C{d['die_c']}"
            f"_R{d['die_r']}"
            f"_{d['band']}"
            f"_Fitting.png"
        )

        fig.savefig(
            os.path.join(
                save_dir,filename
            ),bbox_inches='tight',dpi=150
        )
        plt.close(fig)

        return filename

    except Exception as e:

        print(
            f"오류 발생: "
            f"{d['wafer_id']} "
            f"C{d['die_c']} "
            f"R{d['die_r']} "
            f"{d['band']} "
            f"-> {e}"
        )

        return None
# 실행부

zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = [
    'D07','D08','D23','D24'
]

print("▶ 데이터 파싱을 시작합니다...")

parsed_data_list = list(
    parse_wafer_data(
        zip_path,target_wafers
    )
)

total_items = len(
    parsed_data_list
)

print(
    f"▶ 리플 제거 및 확대 플롯 생성 "
    f"({total_items}개)"
)

success_count = 0

for i, d in enumerate(
    parsed_data_list,1
):

    result = _draw_and_save_zoomed_plot(
        d,base_save_dir
    )

    if result is not None:
        success_count += 1

    if i % 50 == 0:

        print(
            f"진행: "f"{i}/{total_items}"
        )

print(
    f"✅ 리플 제거 및 확대 그래프 저장 완료 "
    f"(총 {success_count}개)"
)

▶ 데이터 파싱을 시작합니다...
▶ 리플 제거 및 확대 플롯 생성 (98개)
진행: 50/98
✅ 리플 제거 및 확대 그래프 저장 완료 (총 98개)


### ⚡ [6] 전압에 따른 위상 변화 분석 (Phase shift - V)
* **파일명**: `Phase shift - V.py`
* **설명**: 인가된 전압($V$)에 따른 광학적 위상 변화($\Delta\phi$) 관계를 추적하여 소자의 전기-광학적 변조 효율을 분석합니다.

In [22]:
from scipy.signal import find_peaks, savgol_filter

def _draw_and_save_phase(d, base_save_dir):

    try:

        m = (
            (d['ref_data']['wl'] >= d['wl_min']) &
            (d['ref_data']['wl'] <= d['wl_max'])
        )

        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        if len(v_ref_wl) < 31:
            return None

        if np.isnan(v_ref_il).all():
            return None

        sm_ref = savgol_filter(
            v_ref_il,31,3
        )

        poly_func = np.poly1d(
            np.polyfit(
                v_ref_wl,sm_ref,3
            )
        )

        tgt_data = next(
            (
                b
                for b in d['bias_data_list']
                if b['bias'] == -2.0
            ),
            d['bias_data_list'][0]
            if d['bias_data_list']
            else None
        )

        if tgt_data is None:
            return None

        w = tgt_data['wl']
        i = tgt_data['il']

        m_t = (
            (w >= d['wl_min']) &
            (w <= d['wl_max'])
        )

        if np.sum(m_t) < 31:
            return None

        flat = (
            savgol_filter(
                i[m_t],31,3
            )- poly_func(w[m_t])
        )

        peaks, _ = find_peaks(
            flat,prominence=3.0,distance=30
        )

        if len(peaks) < 2:
            return None

        w_t = w[m_t]

        centers = (
            w_t[peaks[:-1]]+ w_t[peaks[1:]]
        ) / 2

        idx = np.argmin(
            np.abs(
                centers - d['target_wl']
            )
        )

        if idx + 1 >= len(peaks):
            return None

        FSR = (
            w_t[peaks[idx + 1]]- w_t[peaks[idx]]
        )

        if abs(FSR) < 1e-12:
            return None

        phase_res = []

        for b in d['bias_data_list']:

            if b['bias'] is None:
                continue

            w_b = b['wl']
            i_b = b['il']

            m_b = (
                (w_b >= d['wl_min']) &
                (w_b <= d['wl_max'])
            )

            if np.sum(m_b) < 31:
                continue

            if np.isnan(i_b[m_b]).all():
                continue

            flat_b = (
                savgol_filter(
                    i_b[m_b],31,3
                )- poly_func(w_b[m_b])
            )

            p_b, _ = find_peaks(
                flat_b,prominence=3.0,distance=30
            )

            if len(p_b) < 2:
                continue

            wb_v = w_b[m_b]

            cent = (
                wb_v[p_b[:-1]]+ wb_v[p_b[1:]]
            ) / 2

            i_idx = np.argmin(
                np.abs(
                    cent - d['target_wl']
                )
            )

            if i_idx + 1 >= len(p_b):
                continue

            lp = p_b[i_idx]
            rp = p_b[i_idx + 1]

            v_idx = (
                lp+ np.argmin(
                    flat_b[lp:rp + 1]
                )
            )

            phase_res.append(
                {
                    'bias': b['bias'],
                    'v_wl': wb_v[v_idx]
                }
            )

        if len(phase_res) == 0:
            return None

        phase_res.sort(
            key=lambda x: x['bias']
        )

        z_data = next(
            (
                p
                for p in phase_res
                if abs(p['bias']) < 1e-6
            ),
            None
        )

        if z_data is None:
            return None

        biases = []
        phases = []

        for p in phase_res:

            biases.append(
                p['bias']
            )

            phases.append(
                (
                    (
                        p['v_wl']- z_data['v_wl']
                    )/ FSR
                )* 360.0
            )

        ph_arr = np.rad2deg(
            np.unwrap(
                np.deg2rad(
                    (
                        np.array(phases)+ 180
                    )% 360- 180
                )
            )
        )

        fig, ax = plt.subplots(
            figsize=(10, 6)
        )

        ax.plot(
            biases,ph_arr,
            marker='o',markersize=8,linewidth=2.5
        )

        ax.axhline(
            0,color='gray',
            linestyle='--',linewidth=2,alpha=0.6
        )

        ax.axvline(
            0,color='gray',linestyle='--',
            linewidth=2,alpha=0.6
        )

        ax.set_title(
            f"Wafer: {d['wafer_id']} / "
            f"Coord: ({d['die_c']}, {d['die_r']}) / "
            f"Band: {d['band']}\n"
            f"Phase Shift",fontsize=18,
            fontweight='bold',pad=15
        )

        ax.set_xlabel(
            "Bias Voltage (V)",fontsize=16,
            fontweight='bold'
        )

        ax.set_ylabel(
            "Phase Shift (deg)",
            fontsize=16,fontweight='bold'
        )

        ax.grid(
            True,linestyle='--',alpha=0.6,linewidth=1
        )

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        date_str = d.get(
            'date','Unknown_Date'
        )

        save_dir = os.path.join(
            base_save_dir,d['wafer_id'],date_str
        )

        os.makedirs(
            save_dir,exist_ok=True
        )

        filename = (
            f"{d['wafer_id']}"
            f"_C{d['die_c']}"
            f"_R{d['die_r']}"
            f"_{d['band']}"
            f"_Phase.png"
        )

        fig.savefig(
            os.path.join(
                save_dir,filename
            ),bbox_inches='tight',dpi=150
        )

        plt.close(fig)

        return filename

    except Exception as e:

        print(
            f"오류 발생: "
            f"{d['wafer_id']} "
            f"C{d['die_c']} "
            f"R{d['die_r']} "
            f"{d['band']} "
            f"-> {e}"
        )

        return None
# 실행부
zip_path = "./dat/HY202103"
base_save_dir = "./res/png"
target_wafers = [
    'D07','D08','D23','D24'
]

print("▶ 데이터 파싱을 시작합니다...")

parsed_data_list = list(
    parse_wafer_data(
        zip_path,target_wafers
    )
)

total_items = len(
    parsed_data_list
)

print(
    f"▶ Phase Shift 플롯 생성 "f"({total_items}개)"
)

success_count = 0

for i, d in enumerate(
    parsed_data_list,1
):

    result = _draw_and_save_phase(
        d,base_save_dir
    )

    if result is not None:
        success_count += 1

    if i % 50 == 0:

        print(
            f"진행: "f"{i}/{total_items}"
        )

print(
    f"✅ Phase Shift 저장 완료 "f"(총 {success_count}개)"
)

▶ 데이터 파싱을 시작합니다...
▶ Phase Shift 플롯 생성 (98개)
진행: 50/98
✅ Phase Shift 저장 완료 (총 98개)


### 🔋 [7] VpiL 산출 (VpiL Calculation)
* **파일명**: `VpiL.py`
* **설명**: 반파장 전압(Vpi)과 소자 길이(L)의 곱인 $V_\pi\cdot L$을 계산하여 단위 길이당 변조 효율 지표를 도출합니다.

In [23]:
import os
from scipy.signal import find_peaks, savgol_filter

def q_sub(x, y):

    if len(x) < 3:
        return x[np.argmin(y)]

    idx = np.argmin(y)

    if idx == 0 or idx == len(x) - 1:
        return x[idx]

    c = np.polyfit(
        x[idx - 1:idx + 2],y[idx - 1:idx + 2],2
    )

    if abs(c[0]) < 1e-12:
        return x[idx]

    return -c[1] / (2 * c[0])


def _process_vpil(
    d,base_res_dir,L_length
):

    try:

        wafer = d['wafer_id']
        band = d['band']
        c = d['die_c']
        r = d['die_r']

        date_str = d.get(
            'date','Unknown_Date'
        )

        m = (
            (d['ref_data']['wl'] >= d['wl_min']) &
            (d['ref_data']['wl'] <= d['wl_max'])
        )

        v_ref_wl = d['ref_data']['wl'][m]
        v_ref_il = d['ref_data']['il'][m]

        if len(v_ref_wl) < 31:
            return None

        if np.isnan(v_ref_il).all():
            return None

        sm_ref = savgol_filter(
            v_ref_il,31,3
        )

        poly_func = np.poly1d(
            np.polyfit(
                v_ref_wl,sm_ref,3
            )
        )

        z_data = next(
            (
                b
                for b in d['bias_data_list']
                if b['bias'] == 0.0
            ),
            None
        )

        if z_data is None:
            return None

        m0 = (
            (z_data['wl'] >= d['wl_min']) &
            (z_data['wl'] <= d['wl_max'])
        )

        w0 = z_data['wl'][m0]
        i0 = z_data['il'][m0]

        if len(w0) < 31:
            return None

        flat0 = (
            savgol_filter(i0, 31, 3)
            - poly_func(w0)
        )

        v0, _ = find_peaks(
            -flat0,prominence=0.3,distance=20
        )

        if len(v0) < 2:
            return None

        fsr = np.mean(
            np.diff(
                w0[v0]
            )
        )

        if abs(fsr) < 1e-12:
            return None

        cwl = w0[
            v0[
                np.argmin(
                    np.abs(
                        w0[v0]
                        - d['target_wl']
                    )
                )
            ]
        ]

        volts = []
        p_pi = []

        sh = fsr / 2.5

        for b in d['bias_data_list']:

            if b['bias'] is None:
                continue

            w = b['wl']
            i = b['il']

            mb = (
                (w >= d['wl_min']) &
                (w <= d['wl_max'])
            )

            wb = w[mb]
            ib = i[mb]

            mloc = (
                (wb >= cwl - sh) &
                (wb <= cwl + sh)
            )

            # savgol_filter(window=11) 보호
            if np.sum(mloc) < 11:
                continue

            flat = (
                savgol_filter(
                    ib[mloc],11,3
                )- poly_func(
                    wb[mloc]
                )
            )

            vwl = q_sub(
                wb[mloc],flat
            )

            volts.append(
                b['bias']
            )

            p_pi.append(
                2.0* (vwl - cwl)/ fsr
            )

        if len(volts) < 5:
            return None

        volts = np.array(volts)
        p_pi = np.array(p_pi)

        fit = np.poly1d(
            np.polyfit(
                volts,p_pi,2
            )
        )

        derivative = np.polyder(
            fit
        )

        slope_at_0 = abs(
            derivative(0.0)
        )

        slope_at_0 = max(
            slope_at_0,1e-5
        )

        vpil_0V = (
            L_length/ slope_at_0
        )

        if not (
            0.1<= vpil_0V<= 10.0
        ):
            return None

        fig, ax = plt.subplots(
            figsize=(10, 6)
        )

        vpil_curve = (
            L_length/ np.maximum(
                np.abs(
                    derivative(volts)
                ),1e-5
            )
        )

        ax.plot(
            volts,vpil_curve,'s-',linewidth=2.5,
            markersize=8,color='gray',label='VpiL Curve'
        )

        ax.plot(
            0.0,vpil_0V,'r*',markersize=18,
            label=f'Value @ 0V = {vpil_0V:.3f}'
        )

        ax.axhline(
            vpil_0V,color='red',linestyle=':',
            linewidth=2,alpha=0.6
        )

        ax.axvline(
            0.0,color='red',linestyle=':',
            linewidth=2,alpha=0.6
        )

        ax.set_title(
            f"Wafer: {wafer} / "
            f"Coord: ({c}, {r}) / "
            f"Band: {band}\n"
            f"VπL at 0V",fontsize=18,
            fontweight='bold',pad=15
        )

        ax.set_xlabel(
            "Voltage (V)",fontsize=16,fontweight='bold'
        )

        ax.set_ylabel(
            "Vpi*L (V*cm)",fontsize=16,fontweight='bold'
        )

        ax.grid(
            True,linestyle='--',alpha=0.6,
            linewidth=1
        )

        handles, labels = (
            ax.get_legend_handles_labels()
        )

        if len(handles) > 0:
            ax.legend(
                loc='best',
                prop={
                    'size': 12,'weight': 'bold'
                }
            )

        for spine in ax.spines.values():
            spine.set_linewidth(2)

        fig.tight_layout()

        save_dir = os.path.join(
            base_res_dir,wafer,date_str
        )

        os.makedirs(
            save_dir,exist_ok=True
        )

        filename = (
            f"{wafer}"
            f"_C{c}"
            f"_R{r}"
            f"_{band}"
            f"_VpiL.png"
        )

        fig.savefig(
            os.path.join(
                save_dir,filename
            ),bbox_inches='tight',dpi=150
        )

        plt.close(fig)

        return True

    except Exception as e:

        print(
            f"오류 발생: "
            f"{wafer} "
            f"C{c} "
            f"R{r} "
            f"{band} "
            f"-> {e}"
        )

        return None
# 실행부

zip_path = "./dat/HY202103"
base_res_dir = "./res/png"

target_wafers = [
    'D07','D08','D23','D24'
]

L_length = 0.05

os.makedirs(
    base_res_dir,exist_ok=True
)

print("🚀 0V 기준 개별 다이 그래프 생성을 시작합니다...")

parsed_data_list = list(
    parse_wafer_data(
        zip_path,target_wafers
    )
)

valid_count = 0
total_items = len(parsed_data_list)

for i, d in enumerate(
    parsed_data_list,1
):

    result = _process_vpil(
        d,base_res_dir,L_length
    )

    if result:
        valid_count += 1

    if i % 50 == 0:

        print(
            f"진행: "f"{i}/{total_items}"
        )

print(f"✅ 개별 그래프 저장 완료 "f"(유효 Die: {valid_count}개)")

🚀 0V 기준 개별 다이 그래프 생성을 시작합니다...
진행: 50/98
✅ 개별 그래프 저장 완료 (유효 Die: 73개)


### 🌓 [8] 소멸비 분석 (Extinction Ratio Analysis)
* **파일명**: `ER_Analysis.py`
* **설명**: 최대 투과 조건(Max)과 최소 투과 조건(Min)의 차이인 소멸비(Extinction Ratio, ER)를 산출하여 소자의 온/오프 효율을 평가합니다.

In [24]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

zip_path = "./dat/HY202103"
base_res_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']

print("🚀 평탄화 로직 기반 ER 추출 및 날짜별/Center vs Edge 분석을 시작합니다...")

# 1. WaferMap과 BoxPlot 최상위 경로 설정
wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
os.makedirs(wafer_map_dir, exist_ok=True)
os.makedirs(box_plot_dir, exist_ok=True)

er_data_list = []
count = 0

for d in parse_wafer_data(zip_path, target_wafers):
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]
    if len(v_ref_wl) < 4: continue

    poly_func = np.poly1d(np.polyfit(v_ref_wl, v_ref_il, 3))
    max_er = 0.0

    for b in d['bias_data_list']:
        m_b = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        v_wl, v_il = b['wl'][m_b], b['il'][m_b]
        if len(v_wl) == 0: continue

        flat_il = v_il - poly_func(v_wl)
        peaks, _ = find_peaks(flat_il, prominence=3.0, distance=30)

        if len(peaks) >= 2:
            trend = np.poly1d(np.polyfit(v_wl[peaks], flat_il[peaks], 1))
            final_il = flat_il - trend(v_wl)
        else:
            final_il = flat_il

        er = np.percentile(final_il, 99) - np.percentile(final_il, 1)
        max_er = max(max_er, er)

    er_data_list.append({
        'Wafer': d['wafer_id'], 'Band': d['band'], 'Date': date_str, 'Column': d['die_c'], 'Row': d['die_r'],
        'Radius': np.sqrt(d['die_c'] ** 2 + d['die_r'] ** 2), 'ER': max_er
    })

    count += 1
    if count % 100 == 0: print(f"현재 {count}개 Die 파싱 및 평탄화 ER 추출 완료...")

df = pd.DataFrame(er_data_list)


# 2. 자정을 넘긴 측정 데이터(0603, 0604) 병합 처리

def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str

df['Date'] = df['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.")

filtered_df = pd.DataFrame()

for (wafer, band, date), group in df.groupby(['Wafer', 'Band', 'Date']):
    if len(group) > 5:
        m_val, s_val = group['ER'].mean(), group['ER'].std()
        filtered_df = pd.concat(
            [filtered_df, group[(group['ER'] >= m_val - 3 * s_val) & (group['ER'] <= m_val + 3 * s_val)]])
    else:
        filtered_df = pd.concat([filtered_df, group])

max_rad = filtered_df['Radius'].max()
edge_thresh = max_rad * 0.75
filtered_df['Region'] = np.where(filtered_df['Radius'] > edge_thresh, 'Edge', 'Center')

# [통일된 디자인] Wafer Map 그리기

print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")
for b in filtered_df['Band'].unique():
    band_limits = {'min': np.floor(filtered_df[filtered_df['Band'] == b]['ER'].min()),
                   'max': np.ceil(filtered_df[filtered_df['Band'] == b]['ER'].max())}

    for (w, date), group in filtered_df[filtered_df['Band'] == b].groupby(['Wafer', 'Date']):
        if group.empty: continue

        plt.figure(figsize=(9, 9))
        theta = np.linspace(0, 2 * np.pi, 100)
        plt.plot((max_rad + 0.5) * np.cos(theta), (max_rad + 0.5) * np.sin(theta), color='gray', lw=2)
        plt.plot(edge_thresh * np.cos(theta), edge_thresh * np.sin(theta), color='red', ls='--', lw=2, alpha=0.6)

        sc = plt.scatter(group['Column'], group['Row'], c=group['ER'], cmap='coolwarm_r',
                         vmin=band_limits['min'], vmax=band_limits['max'], s=500, edgecolor='black', alpha=0.9,
                         zorder=5)
        for _, row in group.iterrows():
            plt.text(row['Column'], row['Row'], f"{row['ER']:.1f}", ha='center', va='center', fontsize=10,
                     color='black', fontweight='bold', zorder=10)

        cb = plt.colorbar(sc, shrink=0.8)
        cb.set_label('Extinction Ratio [dB]', fontsize=14, fontweight='bold')
        cb.ax.tick_params(labelsize=12)
        for l in cb.ax.yaxis.get_ticklabels(): l.set_weight("bold")

        plt.title(f"Wafer Map: {w} / {b} (Flattened ER)\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
        plt.axis('off')
        plt.gca().set_aspect('equal')

        # 경로: WaferMap / 웨이퍼 / 날짜
        w_dir = os.path.join(wafer_map_dir, w, date)
        os.makedirs(w_dir, exist_ok=True)
        plt.savefig(os.path.join(w_dir, f"Map_{w}_{b}_{date}_ER.png"), bbox_inches='tight')
        plt.close()

# 3. [통일된 디자인] Box Plot 그리기 (개별 웨이퍼 단위)

print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")
for (w, b, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
    if wbd_df.empty: continue

    plt.figure(figsize=(8, 8))  # 개별 웨이퍼 플롯 사이즈
    pos, data, labels, colors = [], [], [], []

    c = wbd_df[wbd_df['Region'] == 'Center']['ER'].values
    e = wbd_df[wbd_df['Region'] == 'Edge']['ER'].values

    if len(c) > 0:
        pos.append(1)
        data.append(c)
        labels.append(f"Center\nn={len(c)}")
        colors.append('#3498db')
    if len(e) > 0:
        pos.append(2)
        data.append(e)
        labels.append(f"Edge\nn={len(e)}")
        colors.append('#e74c3c')

    if not data: continue

    # Y축 범위를 미리 계산하여 배경색 영역 지정에 사용
    all_data = np.concatenate(data)
    y_min = min(all_data.min(), 15.0) - 2  # 데이터 최소값 또는 15dB 중 작은 값 기준 안전마진
    y_max = max(all_data.max(), 25.0) + 2  # 데이터 최대값 또는 25dB 중 큰 값 기준 안전마진
    plt.ylim(y_min, y_max)

    # 🌟 성능별 배경색(수평 영역) 지정

    # 타겟(20.0) 이상: 연한 초록색 (Good)
    plt.axhspan(20.0, y_max, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')

    # 타겟(20.0) 미만: 연한 붉은/코랄 계열 (Poor) - 눈에 띄면서 부정적인 의미 전달
    plt.axhspan(y_min, 20.0, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')


    box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                      flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                      zorder=2)
    for p, c_hex in zip(box['boxes'], colors): p.set_facecolor(c_hex); p.set_alpha(0.7)

    # Jitter
    for p, d_arr in zip(pos, data):
        plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

    # 평균 및 타겟 라인
    avg_er = wbd_df['ER'].mean()
    plt.axhline(avg_er, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_er:.2f} dB', zorder=4)
    plt.axhline(20.0, color='red', ls='-', lw=2.5, label='Target: 20.00 dB', zorder=4)

    plt.title(f"ER Analysis : {w} ({b})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
    plt.xticks(pos, labels, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.ylabel('Extinction Ratio [dB]', fontsize=16, fontweight='bold')

    plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
    plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
    plt.xlim(0.5, max(pos) + 0.5)
    plt.tight_layout()

    # 경로: BoxPlot / 웨이퍼 / 날짜
    box_dir = os.path.join(box_plot_dir, w, date)
    os.makedirs(box_dir, exist_ok=True)
    plt.savefig(os.path.join(box_dir, f"Box_{w}_{b}_{date}_ER_Flattened.png"), bbox_inches='tight')
    plt.close()

🚀 평탄화 로직 기반 ER 추출 및 날짜별/Center vs Edge 분석을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...


### 📉 [9] 삽입 손실 분석 (Insertion Loss Analysis)
* **파일명**: `IL_Analysis.py`
* **설명**: MZM 소자가 통과시키는 최대 광출력을 기반으로 삽입 손실(Insertion Loss, IL) 값을 계산하고 통계치를 추출합니다.

In [25]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

zip_path = "./dat/HY202103"
base_res_dir = "./res/png"
target_wafers = ['D07', 'D08', 'D23', 'D24']
IL_TARGETS = {'LMZO': -8.75, 'LMZC': -8.75}

# 1. WaferMap과 BoxPlot 최상위 경로 설정
wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
os.makedirs(wafer_map_dir, exist_ok=True)
os.makedirs(box_plot_dir, exist_ok=True)

print("🚀 IL 추출 및 날짜별/Center vs Edge 통합 분석을 시작합니다...")

il_data_list = []
for d in parse_wafer_data(zip_path, target_wafers):
    date_str = d.get('date', 'Unknown_Date')

    max_peak = -999.0
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        m = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        if np.any(m): max_peak = max(max_peak, np.max(b['il'][m]))

    if max_peak != -999.0:
        il_data_list.append(
            {'Wafer': d['wafer_id'], 'Band': d['band'], 'Date': date_str, 'Column': d['die_c'], 'Row': d['die_r'],
             'IL': max_peak})

df = pd.DataFrame(il_data_list).groupby(['Wafer', 'Band', 'Date', 'Column', 'Row'], as_index=False)['IL'].mean()
df['Radius'] = np.sqrt(df['Column'] ** 2 + df['Row'] ** 2)


# 2. 자정을 넘긴 측정 데이터(0603, 0604) 병합 처리

def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str


df['Date'] = df['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.")


filtered_df = pd.DataFrame()
for _, group in df.groupby(['Wafer', 'Band', 'Date']):
    m, s = group['IL'].mean(), group['IL'].std()
    filtered_df = pd.concat([filtered_df, group[(group['IL'] >= m - 3 * s) & (group['IL'] <= m + 3 * s)]])

max_rad = filtered_df['Radius'].max()
filtered_df['Region'] = np.where(filtered_df['Radius'] > max_rad * 0.75, 'Edge', 'Center')

# [통일된 디자인] Wafer Map 그리기

print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")
for b in filtered_df['Band'].unique():
    limits = {'min': np.floor(filtered_df[filtered_df['Band'] == b]['IL'].min()),
              'max': np.ceil(filtered_df[filtered_df['Band'] == b]['IL'].max())}

    for (w, date), grp in filtered_df[filtered_df['Band'] == b].groupby(['Wafer', 'Date']):
        if grp.empty: continue

        plt.figure(figsize=(9, 9))
        th = np.linspace(0, 2 * np.pi, 100)
        plt.plot((max_rad + 0.5) * np.cos(th), (max_rad + 0.5) * np.sin(th), color='gray', lw=2)
        plt.plot(max_rad * 0.75 * np.cos(th), max_rad * 0.75 * np.sin(th), color='red', ls='--', lw=2, alpha=0.6)

        sc = plt.scatter(grp['Column'], grp['Row'], c=grp['IL'], cmap='coolwarm_r', vmin=limits['min'],
                         vmax=limits['max'], s=500, edgecolor='black', alpha=0.9, zorder=5)

        for _, r in grp.iterrows():
            plt.text(r['Column'], r['Row'], f"{r['IL']:.1f}", ha='center', va='center',
                     fontsize=10, fontweight='bold', color='black', zorder=10)

        cb = plt.colorbar(sc, shrink=0.8)
        cb.set_label('IL [dB]', fontsize=14, fontweight='bold')
        cb.ax.tick_params(labelsize=12)
        for l in cb.ax.yaxis.get_ticklabels(): l.set_weight("bold")

        plt.title(f"Wafer Map: {w} / {b} (IL)\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
        plt.axis('off')
        plt.gca().set_aspect('equal')

        # 저장 경로: WaferMap / 웨이퍼 / 날짜
        w_dir = os.path.join(wafer_map_dir, w, date)
        os.makedirs(w_dir, exist_ok=True)
        plt.savefig(os.path.join(w_dir, f"Map_{w}_{b}_{date}_IL.png"), bbox_inches='tight')
        plt.close()

# 3. [통일된 디자인] Box Plot 그리기 (개별 웨이퍼 단위)

print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")
for (w, b, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
    if wbd_df.empty: continue

    plt.figure(figsize=(8, 8))  # 개별 웨이퍼 플롯 사이즈
    pos, data, lbl, clr = [], [], [], []

    c = wbd_df[wbd_df['Region'] == 'Center']['IL'].values
    e = wbd_df[wbd_df['Region'] == 'Edge']['IL'].values

    if len(c) > 0:
        pos.append(1)
        data.append(c)
        lbl.append(f"Center\nn={len(c)}")
        clr.append('#3498db')
    if len(e) > 0:
        pos.append(2)
        data.append(e)
        lbl.append(f"Edge\nn={len(e)}")
        clr.append('#e74c3c')

    if not data: continue

    tgt_il = IL_TARGETS.get(b, -8.75)

    # Y축 범위를 미리 계산하여 배경색 영역 지정에 사용
    all_data = np.concatenate(data)
    y_min = min(all_data.min(), tgt_il) - 2  # 데이터 최소값 또는 타겟 중 작은 값 기준 안전마진
    y_max = max(all_data.max(), tgt_il) + 2  # 데이터 최대값 또는 타겟 중 큰 값 기준 안전마진
    plt.ylim(y_min, y_max)

    # 🌟 성능별 배경색(수평 영역) 지정

    # 타겟(Target) 이상: 연한 초록색 (Good Region)
    plt.axhspan(tgt_il, y_max, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')

    # 타겟(Target) 미만: 연한 붉은/코랄 계열 (Poor Region) - 눈에 띄면서 부정적인 의미 전달
    plt.axhspan(y_min, tgt_il, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')

    box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                      flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                      zorder=2)
    for p, c_hex in zip(box['boxes'], clr): p.set_facecolor(c_hex); p.set_alpha(0.7)

    # Jitter(산점도) 추가
    for p, d_arr in zip(pos, data):
        plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

    avg_il = wbd_df['IL'].mean()

    # 평균 및 타겟 라인
    plt.axhline(avg_il, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_il:.2f} dB', zorder=4)
    plt.axhline(tgt_il, color='red', ls='-', lw=2.5, label=f'Target: {tgt_il:.2f} dB', zorder=4)

    plt.title(f"IL Analysis : {w} ({b})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
    plt.xticks(pos, lbl, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.ylabel('IL [dB]', fontsize=16, fontweight='bold')

    # 범례 설정
    plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
    plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
    plt.xlim(0.5, max(pos) + 0.5)
    plt.tight_layout()

    # 저장 경로: BoxPlot / 웨이퍼 / 날짜
    box_dir = os.path.join(box_plot_dir, w, date)
    os.makedirs(box_dir, exist_ok=True)
    plt.savefig(os.path.join(box_dir, f"Box_{w}_{b}_{date}_IL.png"), bbox_inches='tight')
    plt.close()

🚀 IL 추출 및 날짜별/Center vs Edge 통합 분석을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '0603_0604_Combined'로 통합했습니다.
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...


### 📊 [10] VpiL 통계 분석 (VpiL Analysis)
* **파일명**: `VpiL_Analysis.py`
* **설명**: 산출된 $V_\pi\cdot L$ 데이터들의 웨이퍼 내 분포, 평균, 표준편차 등 전반적인 경향성과 수율을 분석합니다.

In [26]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, savgol_filter
from concurrent.futures import ThreadPoolExecutor

def q_sub(x, y):
    if len(x) < 3: return x[np.argmin(y)]
    idx = np.argmin(y)
    if idx == 0 or idx == len(x) - 1: return x[idx]
    c = np.polyfit(x[idx - 1:idx + 2], y[idx - 1:idx + 2], 2)
    return -c[1] / (2 * c[0]) if abs(c[0]) > 1e-12 else x[idx]


def _extract_vpil_data(args):
    d, L_length = args
    wafer, band, c, r = d['wafer_id'], d['band'], d['die_c'], d['die_r']
    radius = np.sqrt(c ** 2 + r ** 2)
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]

    if len(v_ref_wl) < 31:
        return None

    poly_func = np.poly1d(np.polyfit(v_ref_wl, savgol_filter(v_ref_il, 31, 3), 3))

    # 💡 [수정됨] 부동소수점 오차 방지를 위해 == 0.0 대신 절대값 비교 사용
    z_data = next((b for b in d['bias_data_list'] if b['bias'] is not None and abs(b['bias']) < 1e-3), None)
    if not z_data:
        return None

    m0 = (z_data['wl'] >= d['wl_min']) & (z_data['wl'] <= d['wl_max'])
    w0, i0 = z_data['wl'][m0], z_data['il'][m0]
    flat0 = savgol_filter(i0, 31, 3) - poly_func(w0)
    v0, _ = find_peaks(-flat0, prominence=0.3, distance=20)

    if len(v0) < 2:
        return None

    fsr = np.mean(np.diff(w0[v0]))
    cwl = w0[v0[np.argmin(np.abs(w0[v0] - d['target_wl']))]]

    volts, p_pi = [], []
    sh = fsr / 2.5
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        w, i = b['wl'], b['il']
        mb = (w >= d['wl_min']) & (w <= d['wl_max'])
        wb, ib = w[mb], i[mb]
        mloc = (wb >= cwl - sh) & (wb <= cwl + sh)
        if np.sum(mloc) < 5: continue
        flat = savgol_filter(ib[mloc], 11, 3) - poly_func(wb[mloc])
        vwl = q_sub(wb[mloc], flat)
        volts.append(b['bias'])
        p_pi.append(2.0 * (vwl - cwl) / fsr)

    if len(volts) < 5:
        return None

    volts, p_pi = np.array(volts), np.array(p_pi)

    fit = np.poly1d(np.polyfit(volts, p_pi, 2))
    slope_at_0 = max(np.abs(np.polyder(fit)(0.0)), 1e-5)
    vpil_0V = L_length / slope_at_0

    # 💡 데이터가 안 나온다면 여기가 원인일 수 있습니다! (범위 확인 필요)
    if not (0.1 <= vpil_0V <= 10.0):
        return None

    return {
        'Wafer': wafer, 'Band': band, 'Date': date_str, 'Column': c, 'Row': r,
        'Radius': radius, 'VpiL_0V': vpil_0V
    }


def main():
    zip_path = "./dat/HY202103"
    base_res_dir = "./res/png"
    target_wafers = ['D07', 'D08', 'D23', 'D24']
    L_length = 0.05
    TARGET_VPIL = {'LMZO': 1.4, 'LMZC': 2.0}

    wafer_map_dir = os.path.join(base_res_dir, "WaferMap")
    box_plot_dir = os.path.join(base_res_dir, "BoxPlot")
    os.makedirs(wafer_map_dir, exist_ok=True)
    os.makedirs(box_plot_dir, exist_ok=True)

    print("🚀 분석용 데이터 추출을 시작합니다...")

    parsed_data_list = list(parse_wafer_data(zip_path, target_wafers))
    tasks = [(d, L_length) for d in parsed_data_list]
    summary_rows = []

    with ThreadPoolExecutor(max_workers=8) as ex:
        futures = [ex.submit(_extract_vpil_data, t) for t in tasks]
        for f in futures:
            res = f.result()
            if res is not None:
                summary_rows.append(res)

    if not summary_rows:
        print("❌ 유효한 데이터를 찾지 못했습니다. _extract_vpil_data의 필터 조건을 확인하세요.")
        return

    df = pd.DataFrame(summary_rows)

    def merge_midnight_dates(date_val):
        date_str = str(date_val)
        if '0603' in date_str or '0604' in date_str:
            return '20190603'
        return date_str

    df['Date'] = df['Date'].apply(merge_midnight_dates)
    print("🕒 날짜 병합 처리 완료.")

    filtered_df = pd.DataFrame()
    df_grouped = df.groupby(['Wafer', 'Band', 'Date', 'Column', 'Row', 'Radius'], as_index=False)['VpiL_0V'].mean()

    for (wafer, band, date), group in df_grouped.groupby(['Wafer', 'Band', 'Date']):
        if len(group) > 5:
            m, s = group['VpiL_0V'].mean(), group['VpiL_0V'].std()
            valid_group = group[(group['VpiL_0V'] >= m - 3 * s) & (group['VpiL_0V'] <= m + 3 * s)]
            filtered_df = pd.concat([filtered_df, valid_group])
        else:
            filtered_df = pd.concat([filtered_df, group])

    max_r = filtered_df['Radius'].max()
    edge_limit = max_r * 0.75
    filtered_df['Region'] = np.where(filtered_df['Radius'] > edge_limit, 'Edge', 'Center')

    print(f"✅ 데이터 추출 및 필터링 완료! (총 {len(filtered_df)}개 다이 분석)")

    # Wafer Map 그리기
    print("▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...")
    band_limits = {}
    for b in filtered_df['Band'].unique():
        b_data = filtered_df[filtered_df['Band'] == b]['VpiL_0V']
        band_limits[b] = {'min': b_data.min() - 0.05, 'max': b_data.max() + 0.05}

    for (wafer, band, date), group in filtered_df.groupby(['Wafer', 'Band', 'Date']):
        plt.figure(figsize=(9, 9))
        theta = np.linspace(0, 2 * np.pi, 100)
        plt.plot((max_r + 0.5) * np.cos(theta), (max_r + 0.5) * np.sin(theta), color='gray', lw=2)
        plt.plot(edge_limit * np.cos(theta), edge_limit * np.sin(theta), color='red', ls='--', lw=2, alpha=0.6)

        v_min, v_max = band_limits[band]['min'], band_limits[band]['max']

        scatter = plt.scatter(group['Column'], group['Row'], c=group['VpiL_0V'], cmap='coolwarm',
                              vmin=v_min, vmax=v_max, s=500, edgecolor='black', alpha=0.9, zorder=5)

        for _, row in group.iterrows():
            plt.text(row['Column'], row['Row'], f"{row['VpiL_0V']:.2f}",
                     ha='center', va='center', fontsize=10, weight='bold', color='black', zorder=10)

        cb = plt.colorbar(scatter, shrink=0.8)
        cb.set_label('Vpi*L @ 0V [V*cm]', fontsize=14, fontweight='bold')
        cb.ax.tick_params(labelsize=12)
        for l in cb.ax.yaxis.get_ticklabels(): l.set_weight("bold")

        plt.title(f"Wafer Map: {wafer} / {band} (VpiL)\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
        plt.axis('off')
        plt.gca().set_aspect('equal')

        w_dir = os.path.join(wafer_map_dir, wafer, date)
        os.makedirs(w_dir, exist_ok=True)
        plt.savefig(os.path.join(w_dir, f"Map_{wafer}_{band}_{date}_VpiL_0V.png"), bbox_inches='tight')
        plt.close()

    # Box Plot 그리기 (VpiL 맞춤형 디자인)
    print("▶ 웨이퍼 및 날짜별 Box Plot 생성 중...")

    for (wafer, band, date), wbd_df in filtered_df.groupby(['Wafer', 'Band', 'Date']):
        current_target = TARGET_VPIL.get(band, 1.5)
        avg_vpil = wbd_df['VpiL_0V'].mean()

        plt.figure(figsize=(8, 8))
        pos, data, labels, colors = [], [], [], []

        c_data = wbd_df[wbd_df['Region'] == 'Center']['VpiL_0V'].values
        e_data = wbd_df[wbd_df['Region'] == 'Edge']['VpiL_0V'].values

        if len(c_data) > 0:
            pos.append(1)
            data.append(c_data)
            labels.append(f"Center\nn={len(c_data)}")
            colors.append('#3498db')
        if len(e_data) > 0:
            pos.append(2)
            data.append(e_data)
            labels.append(f"Edge\nn={len(e_data)}")
            colors.append('#e74c3c')

        if not data: continue

        all_data = np.concatenate(data)
        y_min = min(all_data.min(), current_target) - 0.2
        y_max = max(all_data.max(), current_target) + 0.2
        plt.ylim(y_min, y_max)

        # [VpiL 맞춤] 작을수록 좋으므로 Target 아래가 초록색(Good)
        plt.axhspan(y_min, current_target, facecolor='#e8f8f5', alpha=0.6, zorder=0, label='Good Region')
        plt.axhspan(current_target, y_max, facecolor='#fadbd8', alpha=0.6, zorder=0, label='Poor Region')

        box = plt.boxplot(data, positions=pos, patch_artist=True, widths=0.5,
                          flierprops=dict(marker='d', markerfacecolor='black', markersize=6, alpha=0.6),
                          zorder=2)
        for p, c in zip(box['boxes'], colors): p.set_facecolor(c); p.set_alpha(0.7)

        for p, d_arr in zip(pos, data):
            plt.scatter(np.random.normal(p, 0.05, len(d_arr)), d_arr, color='black', alpha=0.5, s=20, zorder=3)

        plt.axhline(avg_vpil, color='blue', ls='--', lw=2.5, label=f'Avg: {avg_vpil:.2f} V*cm', zorder=4)
        plt.axhline(current_target, color='red', ls='-', lw=2.5, label=f'Target: {current_target:.2f} V*cm', zorder=4)

        plt.title(f"VpiL Analysis : {wafer} ({band})\nDate: {date}", fontsize=18, fontweight='bold', pad=15)
        plt.xticks(pos, labels, fontsize=14, fontweight='bold')
        plt.yticks(fontsize=14, fontweight='bold')
        plt.ylabel("Vpi*L @ 0V [V*cm]", fontsize=16, fontweight='bold')

        plt.legend(loc='upper right', prop={'size': 11, 'weight': 'bold'})
        plt.grid(True, axis='y', ls=':', alpha=0.4, zorder=1)
        plt.xlim(0.5, max(pos) + 0.5)
        plt.tight_layout()

        box_dir = os.path.join(box_plot_dir, wafer, date)
        os.makedirs(box_dir, exist_ok=True)
        plt.savefig(os.path.join(box_dir, f"Box_{wafer}_{band}_{date}_VpiL_0V.png"), bbox_inches='tight')
        plt.close()

    print("✅ 모든 그래프가 웨이퍼별/날짜별 폴더 구조로 저장되었습니다!")
if __name__ == "__main__":
    main()

🚀 분석용 데이터 추출을 시작합니다...
🕒 날짜 병합 처리 완료.
✅ 데이터 추출 및 필터링 완료! (총 72개 다이 분석)
▶ 웨이퍼 및 날짜별 Wafer Map 생성 중...
▶ 웨이퍼 및 날짜별 Box Plot 생성 중...
✅ 모든 그래프가 웨이퍼별/날짜별 폴더 구조로 저장되었습니다!


### 🗺️ [11] 통합 시각화: Wafer Map & Box Plot (Combine Plot)
* **파일명**: `combine_plot.py`
* **설명**: 추출된 IL, ER, Vpi 데이터를 기반으로 **Wafer Map**을 그려 수율의 공간적 분포를 확인하고, **Box Plot**을 통해 전체적인 산포와 다이(Die) 간 편차를 한눈에 비교합니다.

In [27]:
import os
import math
from PIL import Image


def get_sort_index(filename):
    """
    다이(Die) 분석 이미지의 정렬 순서를 결정합니다.
    1.Plot(Raw) -> 2.Flatting -> 3.Fitting -> 4.Zoom -> 5.Phase shift -> 6.VpiL
    """
    name = filename.lower()
    if 'raw' in name or 'plot' in name:
        return 1
    elif 'flatting' in name or 'flat' in name:
        return 2
    elif 'zoom' in name:
        return 3
    elif 'fitting' in name or 'fit' in name:
        return 4
    elif 'phase' in name:
        return 5
    elif 'vpil' in name:
        return 6
    else:
        return 99


def get_map_sort_index(filename):
    """
    웨이퍼 맵 및 박스 플롯 이미지의 정렬 순서를 결정합니다.
    요청하신 순서: 1.ER(소광비) -> 2.IL(손실) -> 3.VpiL(효율) 순서로 배치합니다.
    """
    name = filename.upper()
    if 'ER' in name:
        return 1
    elif 'IL' in name and 'VPIL' not in name:
        return 2
    elif 'VPIL' in name:
        return 3
    else:
        return 99


def merge_die_images(base_dir, delete_originals=True):
    """
    각 날짜 폴더 내의 분석 이미지들을 좌표(Die) 및 밴드(Band)별로 그룹화하여
    가로 3, 세로 2 격자 형태로 병합하고 원본을 삭제합니다.
    """
    print("▶ 1. 날짜 폴더 내 다이(Die) 분석 이미지 병합 및 원본 파일 관리를 시작합니다...")
    combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(base_dir):
        wafer_path = os.path.join(base_dir, wafer)

        # 🌟 WaferMap, BoxPlot 등 시스템 폴더는 다이(Die) 병합에서 제외
        if not os.path.isdir(wafer_path) or wafer in ["Analysis", "WaferMap", "BoxPlot"]:
            continue

        for date_folder in os.listdir(wafer_path):
            date_path = os.path.join(wafer_path, date_folder)
            if not os.path.isdir(date_path):
                continue

            for f in os.listdir(date_path):
                if f.startswith('HY202103_') and 'LION1_DCM' in f and f.endswith('.png'):
                    try:
                        os.remove(os.path.join(date_path, f))
                    except:
                        pass

            image_files = [f for f in os.listdir(date_path) if f.endswith('.png')
                           and not f.startswith('Map_')
                           and not f.startswith('Box_')
                           and not f.startswith('HY202103_')]

            if not image_files:
                continue

            die_groups = {}
            for f in image_files:
                parts = f.replace(".png", "").split('_')
                if len(parts) >= 4:
                    try:
                        c_val = parts[1].replace('C', '')
                        r_val = parts[2].replace('R', '')
                        band = parts[3]
                        group_key = (c_val, r_val, band)

                        if group_key not in die_groups:
                            die_groups[group_key] = []
                        die_groups[group_key].append(f)
                    except Exception as e:
                        print(f"파일 이름 분석 오류: {f} -> {e}")

            for (c_val, r_val, band), files in die_groups.items():
                if not files:
                    continue

                files.sort(key=get_sort_index)

                images = []
                valid_files = []
                for img_name in files:
                    try:
                        img_path = os.path.join(date_path, img_name)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass
                if not images:
                    continue

                cols = 3
                rows = math.ceil(len(images) / cols)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                grid_width = cols * max_width
                grid_height = rows * max_height
                new_im = Image.new('RGB', (grid_width, grid_height), color='white')

                for i, img in enumerate(images):
                    x_offset = (i % cols) * max_width
                    y_offset = (i // cols) * max_height
                    new_im.paste(img, (x_offset, y_offset))
                    img.close()

                save_filename = f"HY202103_{wafer}_({c_val},{r_val})_LION1_DCM_{band}.png"
                new_im.save(os.path.join(date_path, save_filename))
                combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 원본 파일 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {combine_count}개의 다이(Die) 요약 이미지 병합 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def merge_wafer_maps(base_dir, delete_originals=True):
    """🌟 WaferMap 폴더 내의 웨이퍼 맵 3종(ER, IL, VpiL)을 1개의 이미지로 가로 병합합니다."""
    wafermap_dir = os.path.join(base_dir, "WaferMap")
    if not os.path.exists(wafermap_dir):
        print(f"  [안내] WaferMap 폴더가 없어 웨이퍼 맵 병합을 건너뜁니다.")
        return

    print("▶ 2. 웨이퍼별 통합 맵(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...")
    map_combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(wafermap_dir):
        w_path = os.path.join(wafermap_dir, wafer)
        if not os.path.isdir(w_path):
            continue

        for date_folder in os.listdir(w_path):
            d_path = os.path.join(w_path, date_folder)
            if not os.path.isdir(d_path):
                continue

            band_images = {}
            for f in os.listdir(d_path):
                if f.startswith('Summary_WaferMap'):
                    try:
                        os.remove(os.path.join(d_path, f))
                    except:
                        pass
                    continue

                if f.startswith('Map_') and f.endswith('.png'):
                    # 파일명 예: Map_D07_LMZO_20190715_IL.png
                    parts = f.replace(f"Map_{wafer}_", "").split("_")
                    band = parts[0]
                    if band not in band_images:
                        band_images[band] = []
                    band_images[band].append(f)

            for band, files in band_images.items():
                if len(files) < 2:
                    continue

                files.sort(key=get_map_sort_index)

                images = []
                valid_files = []
                for f in files:
                    try:
                        img_path = os.path.join(d_path, f)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass

                if not images:
                    continue

                cols = len(images)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                new_im = Image.new('RGB', (max_width * cols, max_height), color='white')
                for i, img in enumerate(images):
                    new_im.paste(img, (i * max_width, 0))
                    img.close()

                save_filename = f"Summary_WaferMap_{wafer}_{band}_{date_folder}.png"
                new_im.save(os.path.join(d_path, save_filename))
                map_combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 웨이퍼 맵 원본 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {map_combine_count}장의 통합 웨이퍼 맵 생성 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 웨이퍼 맵 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def merge_box_plots(base_dir, delete_originals=True):
    """🌟 BoxPlot 폴더 내의 박스 플롯 3종(ER, IL, VpiL)을 1개의 이미지로 가로 병합합니다."""
    boxplot_dir = os.path.join(base_dir, "BoxPlot")
    if not os.path.exists(boxplot_dir):
        print(f"  [안내] BoxPlot 폴더가 없어 박스 플롯 병합을 건너뜁니다.")
        return

    print("▶ 3. 통합 박스 플롯(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...")
    box_combine_count = 0
    deleted_count = 0

    for wafer in os.listdir(boxplot_dir):
        w_path = os.path.join(boxplot_dir, wafer)
        if not os.path.isdir(w_path):
            continue

        for date_folder in os.listdir(w_path):
            d_path = os.path.join(w_path, date_folder)
            if not os.path.isdir(d_path):
                continue

            band_images = {}
            for f in os.listdir(d_path):
                if f.startswith('Summary_BoxPlot'):
                    try:
                        os.remove(os.path.join(d_path, f))
                    except:
                        pass
                    continue

                if f.startswith('Box_') and f.endswith('.png'):
                    # 파일명 예: Box_D07_LMZO_20190715_IL.png
                    parts = f.replace(f"Box_{wafer}_", "").split("_")
                    band = parts[0]
                    if band not in band_images:
                        band_images[band] = []
                    band_images[band].append(f)

            for band, files in band_images.items():
                if len(files) < 2:
                    continue

                files.sort(key=get_map_sort_index)

                images = []
                valid_files = []
                for f in files:
                    try:
                        img_path = os.path.join(d_path, f)
                        img = Image.open(img_path)
                        img.load()
                        images.append(img)
                        valid_files.append(img_path)
                    except:
                        pass

                if not images:
                    continue

                cols = len(images)
                max_width = max(img.size[0] for img in images)
                max_height = max(img.size[1] for img in images)

                new_im = Image.new('RGB', (max_width * cols, max_height), color='white')
                for i, img in enumerate(images):
                    new_im.paste(img, (i * max_width, 0))
                    img.close()

                save_filename = f"Summary_BoxPlot_{wafer}_{band}_{date_folder}.png"
                new_im.save(os.path.join(d_path, save_filename))
                box_combine_count += 1

                if delete_originals:
                    for orig_path in valid_files:
                        try:
                            os.remove(orig_path)
                            deleted_count += 1
                        except Exception as e:
                            print(f"  [경고] 박스 플롯 원본 삭제 실패 ({os.path.basename(orig_path)}): {e}")

    print(f"  ✅ 총 {box_combine_count}장의 통합 박스 플롯 생성 완료!")
    if delete_originals:
        print(f"  🧹 병합에 사용된 박스 플롯 원본 이미지 총 {deleted_count}개 삭제 완료.\n")


def main():
    base_dir = "./res/png"
    base_dir = os.path.abspath(base_dir)

    if not os.path.exists(base_dir):
        print(f"❌ [오류] {base_dir} 경로를 찾을 수 없습니다.")
        return

    # 🌟 원본 파일을 실제로 삭제하려면 True, 원본을 유지하고 싶다면 False로 설정하세요.
    DELETE_ORIGINAL_FILES = True

    # 1. Die 이미지 병합 실행
    merge_die_images(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    # 2. Wafer Map 이미지 병합 실행 (새 경로 적용)
    merge_wafer_maps(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    # 3. Box Plot 이미지 병합 실행 (새로 추가)
    merge_box_plots(base_dir, delete_originals=DELETE_ORIGINAL_FILES)

    print("\n🎉 모든 이미지 병합 및 원본 파일 정리 작업이 성공적으로 마무리되었습니다!")
main()

▶ 1. 날짜 폴더 내 다이(Die) 분석 이미지 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 98개의 다이(Die) 요약 이미지 병합 완료!
  🧹 병합에 사용된 원본 이미지 총 563개 삭제 완료.

▶ 2. 웨이퍼별 통합 맵(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 7장의 통합 웨이퍼 맵 생성 완료!
  🧹 병합에 사용된 웨이퍼 맵 원본 이미지 총 21개 삭제 완료.

▶ 3. 통합 박스 플롯(ER, IL, VpiL) 병합 및 원본 파일 관리를 시작합니다...
  ✅ 총 7장의 통합 박스 플롯 생성 완료!
  🧹 병합에 사용된 박스 플롯 원본 이미지 총 21개 삭제 완료.


🎉 모든 이미지 병합 및 원본 파일 정리 작업이 성공적으로 마무리되었습니다!


### 💾 [12] 분석 결과 요약 및 내보내기 (Export Summary)
* **파일명**: `export_summary.py`
* **설명**: 웨이퍼 내 모든 소자의 분석 결과(IL, ER, Vpi 등)를 종합하여 CSV 또는 Excel 형태의 최종 레포트 파일로 저장합니다.

In [28]:
import os
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# [설정] 아웃라이어(Outlier) 및 에러 판단 기준

THRESHOLD = {
    'IL_MIN': -20.0,
    'ER_MIN': 10.0,
    'VPIL_MIN': 0.2,
    'VPIL_MAX': 3.0
}

zip_path = "./dat/HY202103"

# 저장 디렉토리 분리
save_dir_xlsx = "./res/xlsx"
save_dir_csv = "./res/csv"
os.makedirs(save_dir_xlsx, exist_ok=True)
os.makedirs(save_dir_csv, exist_ok=True)

target_wafers = ['D07', 'D08', 'D23', 'D24']
L_length = 0.05


def q_sub(x, y):
    if len(x) < 3: return x[np.argmin(y)]
    idx = np.argmin(y)
    if idx == 0 or idx == len(x) - 1: return x[idx]
    c = np.polyfit(x[idx - 1:idx + 2], y[idx - 1:idx + 2], 2)
    return -c[1] / (2 * c[0]) if abs(c[0]) > 1e-12 else x[idx]


def check_status(row):
    reasons = []
    if pd.isna(row['IL_dB']):
        reasons.append("IL 데이터 없음")
    elif row['IL_dB'] < THRESHOLD['IL_MIN']:
        reasons.append(f"IL 손실 과다(<{THRESHOLD['IL_MIN']})")

    if pd.isna(row['ER_dB']):
        reasons.append("ER 데이터 없음")
    elif row['ER_dB'] < THRESHOLD['ER_MIN']:
        reasons.append(f"ER 낮음(<{THRESHOLD['ER_MIN']})")

    if pd.isna(row['VpiL_Vcm']):
        reasons.append("VpiL 계산 실패")
    elif row['VpiL_Vcm'] < THRESHOLD['VPIL_MIN'] or row['VpiL_Vcm'] > THRESHOLD['VPIL_MAX']:
        reasons.append(f"VpiL 범위 초과({THRESHOLD['VPIL_MIN']}~{THRESHOLD['VPIL_MAX']})")

    return ("정상", "") if not reasons else ("이상 발생", ", ".join(reasons))


# 엑셀 서식 자동 적용 함수

def apply_excel_style(worksheet, dataframe):
    header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    header_font = Font(color='FFFFFF', bold=True)
    error_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    link_font = Font(color='0563C1', underline='single')
    border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'),
                    bottom=Side(style='thin'))

    col_idx = {col: i for i, col in enumerate(dataframe.columns, 1)}

    for col_num, column_title in enumerate(dataframe.columns, 1):
        cell = worksheet.cell(row=1, column=col_num)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')

        for row_num in range(2, len(dataframe) + 2):
            data_cell = worksheet.cell(row=row_num, column=col_num)
            data_cell.border = border

            # 1. Folder_Link 열 (개별 다이 통합 PNG 열기)
            if column_title == 'Folder_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "🖼️ 개별 다이 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            # 2. 웨이퍼 맵 하이퍼링크 (웨이퍼 맵 통합 이미지 열기)
            elif column_title == 'Map_Image_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "🗺️ 웨이퍼 맵 통합 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            # 3. 박스 플롯 하이퍼링크 (박스 플롯 통합 이미지 열기)
            elif column_title == 'BoxPlot_Image_Link':
                target_image = data_cell.value
                if target_image:
                    data_cell.value = f'=HYPERLINK("{target_image}", "📊 박스 플롯 통합 이미지 보기")'
                    data_cell.font = link_font
                    data_cell.alignment = Alignment(horizontal='center')

            else:
                if column_title in ['IL_dB', 'ER_dB', 'VpiL_Vcm'] and isinstance(data_cell.value, (int, float)):
                    data_cell.number_format = '0.000'
                elif column_title == 'R_Squared' and isinstance(data_cell.value, (int, float)):
                    data_cell.number_format = '0.0000'

            # 에러 행 하이라이트
            if 'Status' in col_idx:
                status_value = worksheet.cell(row=row_num, column=col_idx['Status']).value
                if status_value == "이상 발생" and column_title not in ['Folder_Link', 'Map_Image_Link', 'BoxPlot_Image_Link']:
                    data_cell.fill = error_fill

    for col in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value is not None:
                text = str(cell.value)
                max_length = max(max_length, sum(2 if ord(c) > 127 else 1 for c in text))
        worksheet.column_dimensions[column_letter].width = min(max_length + 3, 50)
# 메인 실행 블록

print("🚀 데이터 분석 및 계산을 시작합니다...")
data_list = []

for d in parse_wafer_data(zip_path, target_wafers):
    wafer, band, c, r = d['wafer_id'], d['band'], d['die_c'], d['die_r']
    radius = np.sqrt(c ** 2 + r ** 2)
    date_str = d.get('date', 'Unknown_Date')

    m = (d['ref_data']['wl'] >= d['wl_min']) & (d['ref_data']['wl'] <= d['wl_max'])
    v_ref_wl, v_ref_il = d['ref_data']['wl'][m], d['ref_data']['il'][m]
    if len(v_ref_wl) < 31: continue

    sm_ref = savgol_filter(v_ref_il, 31, 3)
    poly_func = np.poly1d(np.polyfit(v_ref_wl, sm_ref, 3))
    y_fitted = poly_func(v_ref_wl)

    ss_res = np.sum((sm_ref - y_fitted) ** 2)
    ss_tot = np.sum((sm_ref - np.mean(sm_ref)) ** 2)
    r_squared = 1.0 if ss_tot == 0 else 1 - (ss_res / ss_tot)

    max_peak = float('-inf')
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        mb = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        if np.any(mb):
            max_peak = max(max_peak, np.max(b['il'][mb]))
    il_val = max_peak if max_peak != float('-inf') else np.nan

    max_er = 0.0
    for b in d['bias_data_list']:
        if b['bias'] is None: continue
        mb = (b['wl'] >= d['wl_min']) & (b['wl'] <= d['wl_max'])
        v_wl, v_il = b['wl'][mb], b['il'][mb]
        if len(v_wl) == 0: continue
        flat_il = v_il - poly_func(v_wl)
        peaks, _ = find_peaks(flat_il, prominence=3.0, distance=30)
        final_il = flat_il - np.poly1d(np.polyfit(v_wl[peaks], flat_il[peaks], 1))(v_wl) if len(peaks) >= 2 else flat_il
        max_er = max(max_er, np.percentile(final_il, 99) - np.percentile(final_il, 1))
    er_val = max_er if max_er > 0 else np.nan

    vpil_val = np.nan
    z_data = next((b for b in d['bias_data_list'] if b['bias'] == 0.0), None)
    if z_data:
        m0 = (z_data['wl'] >= d['wl_min']) & (z_data['wl'] <= d['wl_max'])
        w0, i0 = z_data['wl'][m0], z_data['il'][m0]
        v0, _ = find_peaks(-(savgol_filter(i0, 31, 3) - poly_func(w0)), prominence=0.3, distance=20)
        if len(v0) >= 2:
            fsr, cwl = np.mean(np.diff(w0[v0])), w0[v0[np.argmin(np.abs(w0[v0] - d['target_wl']))]]
            volts, p_pi = [], []
            for b in d['bias_data_list']:
                if b['bias'] is None: continue
                w, i = b['wl'], b['il']
                mb = (w >= d['wl_min']) & (w <= d['wl_max'])
                wb, ib = w[mb], i[mb]
                mloc = (wb >= cwl - fsr / 2.5) & (wb <= cwl + fsr / 2.5)
                if np.sum(mloc) >= 5:
                    volts.append(b['bias'])
                    p_pi.append(
                        2.0 * (q_sub(wb[mloc], savgol_filter(ib[mloc], 11, 3) - poly_func(wb[mloc])) - cwl) / fsr)
            if len(volts) >= 5:
                vpil_0V = L_length / max(np.abs(np.polyder(np.poly1d(np.polyfit(volts, p_pi, 2)))(0.0)), 1e-5)
                if 0.1 <= vpil_0V <= 10.0: vpil_val = vpil_0V

    data_list.append({
        'Wafer': wafer, 'Band': band, 'Date': date_str, 'Column': c, 'Row': r, 'Radius': radius,
        'IL_dB': il_val, 'ER_dB': er_val, 'VpiL_Vcm': vpil_val, 'R_Squared': r_squared
    })

# --- 전체 데이터 정리 및 날짜 병합 ---
df_full = pd.DataFrame(data_list).groupby(['Wafer', 'Band', 'Date', 'Column', 'Row', 'Radius'], as_index=False).min(
    numeric_only=True)
df_full['Status'], df_full['Reason'] = zip(*df_full.apply(check_status, axis=1))


# [추가된 부분] 0603, 0604 데이터 병합
def merge_midnight_dates(date_val):
    date_str = str(date_val)
    if '0603' in date_str or '0604' in date_str:
        return '20190603'
    return date_str


df_full['Date'] = df_full['Date'].apply(merge_midnight_dates)
print("🕒 0603 및 0604 날짜 데이터를 '20190603'로 통합했습니다.")


# 🌟 1. 개별 다이 이미지 경로 생성
def generate_die_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    c = int(row['Column'])
    r = int(row['Row'])
    band = str(row['Band'])
    base_dir = f"..\\png\\{wafer}\\{date}"
    image_name = f"HY202103_{wafer}_({c},{r})_LION1_DCM_{band}.png"
    return f"{base_dir}\\{image_name}"


df_full['Folder_Link'] = df_full.apply(generate_die_paths, axis=1)

print("--------------------------------------------------")
print("▶ 결과 파일 저장을 시작합니다...")

# [순수 CSV 파일 생성] (링크 컬럼 제거됨, 합본 포함)

csv_df = df_full.drop(columns=['Folder_Link'], errors='ignore')

for wafer_id in csv_df['Wafer'].unique():
    wafer_csv = csv_df[csv_df['Wafer'] == wafer_id]
    csv_file_path = os.path.join(save_dir_csv, f"{wafer_id}_Process_result.csv")
    wafer_csv.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
    print(f"  - 개별 순수 CSV 저장 완료: {csv_file_path}")

total_csv_path = os.path.join(save_dir_csv, "Total_Process_result.csv")
csv_df.to_csv(total_csv_path, index=False, encoding='utf-8-sig')
print(f"  - 🌟 전체 통합 CSV 저장 완료: {total_csv_path}")

print("--------------------------------------------------")

# [엑셀 XLSX 파일 생성] (병합 이미지 링크 포함)

for wafer_id in df_full['Wafer'].unique():
    wafer_df = df_full[df_full['Wafer'] == wafer_id].copy()
    wafer_file_path = os.path.join(save_dir_xlsx, f"{wafer_id}_Process_result.xlsx")
    with pd.ExcelWriter(wafer_file_path, engine='openpyxl') as writer:
        wafer_df.to_excel(writer, index=False, sheet_name='Analysis_Result')
        apply_excel_style(writer.sheets['Analysis_Result'], wafer_df)
    print(f"  - 개별 XLSX 저장 완료: {wafer_file_path}")

total_xlsx_path = os.path.join(save_dir_xlsx, "Total_Process_result.xlsx")
with pd.ExcelWriter(total_xlsx_path, engine='openpyxl') as writer:
    df_full.to_excel(writer, index=False, sheet_name='Analysis_Result')
    apply_excel_style(writer.sheets['Analysis_Result'], df_full)
print(f"  - 🌟 전체 통합 XLSX 저장 완료: {total_xlsx_path}")

# 3. 새로운 Analysis.xlsm 및 Analysis.csv 생성 (통합 이미지 직접 링크)

df_index = df_full[['Wafer', 'Date', 'Band']].drop_duplicates().reset_index(drop=True)


# 🌟 웨이퍼 맵 통합 파일 경로 링크 생성
def generate_map_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    band = str(row['Band'])
    return f"..\\png\\WaferMap\\{wafer}\\{date}\\Summary_WaferMap_{wafer}_{band}_{date}.png"


# 🌟 박스 플롯 통합 파일 경로 링크 생성
def generate_boxplot_paths(row):
    wafer = str(row['Wafer'])
    date = str(row['Date'])
    band = str(row['Band'])
    return f"..\\png\\BoxPlot\\{wafer}\\{date}\\Summary_BoxPlot_{wafer}_{band}_{date}.png"


# 링크 적용 (이름 변경)
df_index['Map_Image_Link'] = df_index.apply(generate_map_paths, axis=1)
df_index['BoxPlot_Image_Link'] = df_index.apply(generate_boxplot_paths, axis=1)

# Analysis.csv 저장 (경로 제거)
analysis_csv_path = os.path.join(save_dir_csv, "Analysis.csv")
df_index.drop(columns=['Map_Image_Link', 'BoxPlot_Image_Link'], errors='ignore').to_csv(analysis_csv_path,
                                                                                          index=False,
                                                                                          encoding='utf-8-sig')
print(f"  - 🌟 마스터 CSV 저장 완료: {analysis_csv_path}")

# Analysis.xlsm 저장 (대시보드 형태)
total_file_path = os.path.join(save_dir_xlsx, "Analysis.xlsm")
with pd.ExcelWriter(total_file_path, engine='openpyxl') as writer:
    df_index.to_excel(writer, index=False, sheet_name='Dashboard_Links')
    apply_excel_style(writer.sheets['Dashboard_Links'], df_index)
print(f"  - 🌟 마스터 XLSX 저장 완료 (통합 이미지 다이렉트 링크 연결 완료): {total_file_path}")

print(f"✅ 모든 분석 및 파일 생성이 성공적으로 완료되었습니다!")

🚀 데이터 분석 및 계산을 시작합니다...
🕒 0603 및 0604 날짜 데이터를 '20190603'로 통합했습니다.
--------------------------------------------------
▶ 결과 파일 저장을 시작합니다...
  - 개별 순수 CSV 저장 완료: ./res/csv\D07_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D08_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D23_Process_result.csv
  - 개별 순수 CSV 저장 완료: ./res/csv\D24_Process_result.csv
  - 🌟 전체 통합 CSV 저장 완료: ./res/csv\Total_Process_result.csv
--------------------------------------------------
  - 개별 XLSX 저장 완료: ./res/xlsx\D07_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D08_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D23_Process_result.xlsx
  - 개별 XLSX 저장 완료: ./res/xlsx\D24_Process_result.xlsx
  - 🌟 전체 통합 XLSX 저장 완료: ./res/xlsx\Total_Process_result.xlsx
  - 🌟 마스터 CSV 저장 완료: ./res/csv\Analysis.csv
  - 🌟 마스터 XLSX 저장 완료 (통합 이미지 다이렉트 링크 연결 완료): ./res/xlsx\Analysis.xlsm
✅ 모든 분석 및 파일 생성이 성공적으로 완료되었습니다!
